# ⚾ 상황 매칭 대조군 실험: 주자·아웃·이닝·카운트를 맞춰도 장타 후 구종 회피가 남는가?

* **작성자:** DY
* **작성일:** 2026-09-26
* **목적:**
  * 지금까지 쓰던 대조군(`field_out`을 시즌 × 구종계열로 층화추출)은 투수·이닝·카운트·주자 상황까지 맞춘 것은 아닙니다. 장타는 주자가 있는 위기 상황에서 더 자주 나올 수 있어, "장타를 맞아서 구종을 바꾼다"는 효과와 "위기 상황이라 구종을 바꾼다"는 효과가 섞였을 가능성이 있습니다.
  * 팀 공유 research CSV의 `on_1b/on_2b/on_3b`(투구 직전 주자)로 대조군을 상황별로 다시 뽑고(M0 → M1 → M2), 순수 감소와 혼합모델 `group` 오즈비가 어떻게 변하는지 봅니다.
* **주의:** CSV에는 점수가 없어 점수차는 매칭과 모델에서 모두 뺐습니다. 점수차를 통제하지 못한 것이 이 실험의 한계입니다.

In [1]:
# 1. 경로 설정 (src 폴더 접근용)
import sys, os
sys.path.append(os.path.abspath('..'))

# 2. Autoreload 설정 (src 내부의 py 파일 수정 시 즉시 반영)
%load_ext autoreload
%autoreload 2

# 경고 메시지 무시 (선택 사항)
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display

from src.analysis.avoidance_stats import match_treatment_to_control, summarize_diff_in_diff
from src.analysis.glmer_runner import check_convergence, extract_fixed_effects_table, fit_glmer
from src.analysis.mixed_effects_model import build_combined_model_dataset
from src.analysis.run_same_batter_rematch_analysis import PLACEBO_SEED, build_placebo_rematch, build_xbh_rematch
from src.analysis.avoidance_stats import compute_season_usage_rate
from src.preprocessing.build_next_ab_dataset import identify_extra_base_hit_events
from src.preprocessing.research_sample import KEY_COLUMNS, list_research_csvs

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

Error importing in API mode: ImportError("dlopen(/Users/dongyunkwak/GitHub/9th-first-project-baseball-2/.venv/lib/python3.14/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib\n  Referenced from: <9F0E20C3-D782-31B5-943E-66F62BCE77E8> /Users/dongyunkwak/GitHub/9th-first-project-baseball-2/.venv/lib/python3.14/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file)")


Trying to import in ABI mode.


## 1. 팀 CSV 읽기

`data/raw`를 쓰지 않고 팀 research CSV 6개를 그대로 읽습니다. CSV에는 점수(`home_score`, `away_score`)가 없어서,
기존 이벤트 빌더가 요구하는 점수 컬럼을 0으로 채워 호출하고 결과의 `score_diff`는 바로 버립니다(어디에도 쓰지 않습니다).

In [2]:
LOAD_COLUMNS = [
    'game_pk', 'game_date', 'at_bat_number', 'pitch_number', 'pitcher', 'player_name', 'batter',
    'stand', 'p_throws', 'pitch_type', 'events', 'balls', 'strikes', 'outs_when_up',
    'inning', 'inning_topbot', 'on_1b', 'on_2b', 'on_3b',
]


def load_team_csv() -> pd.DataFrame:
    csv_files = list_research_csvs()  # data/research/ (형식이 다른 파일명이 있으면 에러)
    frames = [pd.read_csv(path, usecols=LOAD_COLUMNS, encoding='utf-8-sig', low_memory=False) for path in csv_files]
    pitches = pd.concat(frames, ignore_index=True)
    pitches['season'] = pd.to_datetime(pitches['game_date']).dt.year
    pitches['home_score'] = 0  # CSV에 점수 없음: 빌더 호출용 자리 채움 (score_diff는 곧 버림)
    pitches['away_score'] = 0
    return pitches


pitches = load_team_csv()
print(f'{len(pitches):,}행, 투수 {pitches["pitcher"].nunique():,}명, 시즌 {sorted(pitches["season"].unique().tolist())}')

3,592,302행, 투수 1,067명, 시즌 [2021, 2022, 2023, 2024, 2025, 2026]


In [3]:
assert len(pitches) == 3_592_302, len(pitches)
assert pitches['pitcher'].nunique() == 1_067
assert not pitches.duplicated(KEY_COLUMNS).any(), 'CSV 키(game_pk, at_bat_number, pitch_number, pitcher)가 중복'
assert pitches['outs_when_up'].isin([0, 1, 2]).all()
assert pitches[['inning', 'balls', 'strikes']].notna().all().all()
print('표본 검증 통과')

표본 검증 통과


## 2. 상황 변수

이벤트를 만든 투구 시점(투구 직전)의 주자·아웃·이닝·카운트로 매칭용 변수를 만듭니다.
`bases`는 1루=1, 2루=2, 3루=4를 더한 0–7, `base_out_state`는 `bases × 3 + 아웃`(0–23)입니다.

In [4]:
SITUATION_COLUMNS = ['runners_n', 'risp', 'bases', 'base_out_state', 'inning_bucket', 'count_bucket']


def add_situation_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    on1, on2, on3 = out['on_1b'].notna(), out['on_2b'].notna(), out['on_3b'].notna()
    out['runners_n'] = on1.astype(int) + on2.astype(int) + on3.astype(int)
    out['risp'] = (on2 | on3).astype(int)
    out['bases'] = on1.astype(int) + 2 * on2.astype(int) + 4 * on3.astype(int)
    out['base_out_state'] = out['bases'] * 3 + out['outs_when_up'].astype(int)
    out['inning_bucket'] = np.select([out['inning'] <= 3, out['inning'] <= 6], ['1-3', '4-6'], default='7+')
    out['count_bucket'] = np.select(
        [out['strikes'] > out['balls'], out['strikes'] < out['balls']], ['pitcher', 'batter'], default='even'
    )
    return out

In [5]:
_t = pd.DataFrame({
    'on_1b': [np.nan, 1.0, np.nan, 5.0], 'on_2b': [np.nan, np.nan, 2.0, 6.0], 'on_3b': [np.nan, np.nan, np.nan, 7.0],
    'outs_when_up': [0, 1, 2, 2], 'inning': [1, 4, 9, 7], 'balls': [0, 2, 1, 3], 'strikes': [0, 1, 2, 1],
})
_r = add_situation_columns(_t)
assert _r['runners_n'].tolist() == [0, 1, 1, 3]
assert _r['risp'].tolist() == [0, 0, 1, 1]
assert _r['bases'].tolist() == [0, 1, 2, 7]
assert _r['base_out_state'].tolist() == [0, 4, 8, 23]
assert _r['inning_bucket'].tolist() == ['1-3', '4-6', '7+', '7+']
assert _r['count_bucket'].tolist() == ['even', 'batter', 'pitcher', 'batter']
print('add_situation_columns 테스트 통과')

add_situation_columns 테스트 통과


In [6]:
pitches = add_situation_columns(pitches)
print(pitches[['runners_n', 'risp', 'base_out_state']].describe().round(3))
print(pitches['inning_bucket'].value_counts(normalize=True).round(3).to_dict())
print(pitches['count_bucket'].value_counts(normalize=True).round(3).to_dict())

         runners_n         risp  base_out_state
count  3592302.000  3592302.000     3592302.000
mean         0.580        0.237           4.109
std          0.775        0.425           5.291
min          0.000        0.000           0.000
25%          0.000        0.000           1.000
50%          0.000        0.000           2.000
75%          1.000        0.000           5.000
max          3.000        1.000          23.000
{'1-3': 0.362, '4-6': 0.341, '7+': 0.297}
{'even': 0.44, 'pitcher': 0.296, 'batter': 0.264}


## 3. 장타 재대결 이벤트

장타(2루타·3루타·홈런)를 맞은 뒤, 같은 타자와의 재대결 타석이 있는 이벤트를 만듭니다(같은 경기, 같은 투수, 그 타자의 다음 타석).
이 건수는 점수 컬럼과 무관하게 정해지므로, 점수를 채워 넣은 CSV만으로 계산해도 표본이 예상한 크기와 일치하는지 확인합니다.

In [7]:
season_usage = compute_season_usage_rate(pitches)
xbh_all = identify_extra_base_hit_events(pitches)
assert len(xbh_all) == 70_319, len(xbh_all)

xbh = build_xbh_rematch(pitches, season_usage).drop(columns='score_diff')
n_eligible = int(xbh['pitch_family'].notna().sum())
assert len(xbh) == 27_391, len(xbh)
assert n_eligible == 27_371, n_eligible
print(f'장타 {len(xbh_all):,}건, 같은 타자 재대결 {len(xbh):,}건 (구종 있는 {n_eligible:,}건)')

2026-09-27 17:25:15,374 [INFO] 장타 70319건 중 같은 타자·같은 투수 재대결 27391건 (39.0%)
        xbh_total  same_batter_rematch  rematch_share
season                                               
2021        12709                 4688         0.3689
2022        12186                 4623         0.3794
2023        13137                 5280         0.4019
2024        12427                 4977         0.4005
2025        12394                 4916         0.3966
2026         7466                 2907         0.3894


장타 70,319건, 같은 타자 재대결 27,391건 (구종 있는 27,371건)


In [8]:
PITCH_KEYS = ['game_pk', 'at_bat_number', 'pitch_number', 'pitcher']
NEXT_COLUMNS = ['runners_n_next', 'risp_next', 'outs_next']


def attach_event_situation(events: pd.DataFrame, pitches_sit: pd.DataFrame) -> pd.DataFrame:
    """이벤트를 만든 투구의 상황 변수를 이벤트 표에 붙인다 (장타·대조군 공통)."""
    events = events.drop(columns=[c for c in SITUATION_COLUMNS if c in events.columns])
    situation = pitches_sit[PITCH_KEYS + SITUATION_COLUMNS].rename(columns={'pitch_number': 'event_pitch_number'})
    out = events.merge(
        situation, on=['game_pk', 'at_bat_number', 'event_pitch_number', 'pitcher'], how='left', validate='many_to_one'
    )
    assert out[SITUATION_COLUMNS].notna().all().all(), '이벤트 투구를 CSV에서 찾지 못함'
    return out


def attach_next_situation(events: pd.DataFrame, pitches_sit: pd.DataFrame) -> pd.DataFrame:
    """재대결 타석에서 같은 투수가 던진 첫 투구 시점의 주자·아웃을 붙인다 (균형 점검·보정용)."""
    events = events.drop(columns=[c for c in NEXT_COLUMNS if c in events.columns])
    first = (
        pitches_sit.sort_values(['game_pk', 'at_bat_number', 'pitcher', 'pitch_number'])
        .groupby(['game_pk', 'at_bat_number', 'pitcher'], as_index=False)
        .first()[['game_pk', 'at_bat_number', 'pitcher', 'runners_n', 'risp', 'outs_when_up']]
        .rename(columns={
            'at_bat_number': 'next_at_bat_number', 'runners_n': 'runners_n_next',
            'risp': 'risp_next', 'outs_when_up': 'outs_next',
        })
    )
    first['next_at_bat_number'] = first['next_at_bat_number'].astype(float)
    return events.merge(first, on=['game_pk', 'next_at_bat_number', 'pitcher'], how='left', validate='many_to_one')

In [9]:
_ev = pd.DataFrame({
    'game_pk': [1, 1], 'pitcher': [10, 10], 'at_bat_number': [5, 9],
    'event_pitch_number': [3, 2], 'next_at_bat_number': [9.0, 14.0],
})
_px = pd.DataFrame({
    'game_pk': [1] * 5, 'pitcher': [10] * 5, 'at_bat_number': [5, 5, 9, 9, 14], 'pitch_number': [1, 3, 1, 2, 4],
    'runners_n': [0, 2, 1, 1, 3], 'risp': [0, 1, 0, 0, 1], 'bases': [0, 5, 1, 1, 7],
    'base_out_state': [0, 15, 3, 3, 23], 'inning_bucket': ['1-3'] * 5, 'count_bucket': ['even'] * 5,
    'outs_when_up': [0, 0, 1, 1, 2],
})
_a = attach_event_situation(_ev, _px)
assert _a['runners_n'].tolist() == [2, 1] and _a['risp'].tolist() == [1, 0]
_b = attach_next_situation(_a, _px)
assert _b['runners_n_next'].tolist() == [1, 3]
assert _b['risp_next'].tolist() == [0, 1]
assert _b['outs_next'].tolist() == [1, 2]
assert len(_b) == 2
print('attach_*_situation 테스트 통과')

attach_*_situation 테스트 통과


In [10]:
xbh = attach_next_situation(attach_event_situation(xbh, pitches), pitches)
assert len(xbh) == 27_391
display(xbh[['runners_n', 'risp', 'outs_when_up', 'runners_n_next', 'risp_next']].mean().round(3).to_frame('장타 평균'))

,장타 평균
runners_n,0.486
risp,0.185
outs_when_up,0.877
runners_n_next,0.578
risp_next,0.223


## 4. 도우미: 균형표(SMD), 순수 감소, 주자 층

* `smd`: 표준화 평균차(풀링 표준편차). |SMD| < 0.1이면 균형이 맞다고 봅니다.
* `pure_reduction`: `diff = baseline − same_type_share`의 장타 평균 − 대조군 평균 (기존 분석과 같은 정의)
* `runner_group`: 서로 겹치지 않는 3단계 (주자 없음 / 1루만 / 득점권)

In [11]:
def smd(x: pd.Series, y: pd.Series) -> float:
    x, y = x.dropna(), y.dropna()
    pooled = np.sqrt((x.var(ddof=1) + y.var(ddof=1)) / 2)
    diff = x.mean() - y.mean()
    if pooled == 0:
        return 0.0 if diff == 0 else float('inf')
    return diff / pooled


def balance_table(xbh_: pd.DataFrame, placebo_: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = {
        c: {'xbh_mean': xbh_[c].mean(), 'placebo_mean': placebo_[c].mean(), 'smd': smd(xbh_[c], placebo_[c])}
        for c in columns
    }
    return pd.DataFrame.from_dict(rows, orient='index')


def usable(events: pd.DataFrame, baseline_col: str) -> pd.DataFrame:
    d = events[events['has_next_ab'] & events[baseline_col].notna()].copy()
    d['diff'] = d[baseline_col] - d['same_type_share']
    return d


def pure_reduction(xbh_: pd.DataFrame, placebo_: pd.DataFrame, baseline_col: str) -> dict:
    x, p = usable(xbh_, baseline_col), usable(placebo_, baseline_col)
    r = summarize_diff_in_diff(x['diff'], p['diff'])
    return {
        'n_xbh': r['n_treatment'], 'n_placebo': r['n_control'], 'net': r['net_effect'],
        'ci_low': r['ci_low'], 'ci_high': r['ci_high'],
    }


def runner_group(events: pd.DataFrame) -> np.ndarray:
    return np.select([events['risp'] == 1, events['runners_n'] >= 1], ['득점권', '1루만'], default='주자 없음')

In [12]:
assert np.isclose(smd(pd.Series([0.0, 2.0]), pd.Series([1.0, 3.0])), -1 / np.sqrt(2))
assert smd(pd.Series([1.0, 1.0]), pd.Series([1.0, 1.0])) == 0.0
_bt = balance_table(pd.DataFrame({'a': [0.0, 2.0]}), pd.DataFrame({'a': [1.0, 3.0]}), ['a'])
assert list(_bt.columns) == ['xbh_mean', 'placebo_mean', 'smd'] and np.isclose(_bt.loc['a', 'smd'], -1 / np.sqrt(2))

_x = pd.DataFrame({'has_next_ab': [True, True, False], 'b': [0.5, 0.5, 0.5], 'same_type_share': [0.3, 0.1, 0.0]})
_p = pd.DataFrame({'has_next_ab': [True, True], 'b': [0.5, 0.5], 'same_type_share': [0.5, 0.3]})
_r = pure_reduction(_x, _p, 'b')
assert _r['n_xbh'] == 2 and _r['n_placebo'] == 2 and np.isclose(_r['net'], 0.2), _r

_g = runner_group(pd.DataFrame({'risp': [0, 0, 1, 1], 'runners_n': [0, 1, 1, 2]}))
assert _g.tolist() == ['주자 없음', '1루만', '득점권', '득점권']
print('균형·순수 감소 도우미 테스트 통과')

균형·순수 감소 도우미 테스트 통과


## 5. 매칭 사다리 (M0 → M1 → M2)

| 단계 | 대조군을 맞추는 칸 |
|---|---|
| M0 | 시즌 × 구종계열 (현행) |
| M1 | M0 + 주자·아웃 24가지 (`base_out_state`) |
| M2 | M1 + 이닝 구간 + 카운트 구간 |

대조군은 기존 `build_placebo_rematch`(칸 지정 가능)로 뽑고, `match_treatment_to_control`로 장타를 대조군 수에 맞춰
칸마다 양쪽 크기를 같게 합니다. 점수차는 CSV에 없어 칸에 넣지 못했습니다.

In [13]:
STEPS = {
    'M0': ('season', 'pitch_family'),
    'M1': ('season', 'pitch_family', 'base_out_state'),
    'M2': ('season', 'pitch_family', 'base_out_state', 'inning_bucket', 'count_bucket'),
}


def build_step(step: str, pitches_sit: pd.DataFrame, xbh_: pd.DataFrame, season_usage_: pd.DataFrame):
    """단계별 대조군을 뽑고 장타를 칸별 대조군 수에 맞춘다. (장타, 대조군) 반환."""
    strata = list(STEPS[step])
    placebo = build_placebo_rematch(pitches_sit, xbh_, season_usage_, strata_cols=tuple(strata)).drop(columns='score_diff')
    placebo = attach_next_situation(attach_event_situation(placebo, pitches_sit), pitches_sit)
    xbh_m = match_treatment_to_control(xbh_, placebo, tuple(strata), seed=PLACEBO_SEED)
    same = xbh_m.groupby(strata).size().sort_index().equals(placebo.groupby(strata).size().sort_index())
    assert same, f'{step}: 칸별 표본 크기가 양쪽에서 다름'
    return xbh_m, placebo


pairs = {}
for step in STEPS:
    xbh_m, placebo_m = build_step(step, pitches, xbh, season_usage)
    pairs[step] = (xbh_m, placebo_m)
    print(f'{step}: 장타 {len(xbh_m):,}/{n_eligible:,}건 유지 ({len(xbh_m) / n_eligible:.1%}), 대조군 {len(placebo_m):,}건')

M0: 장타 27,371/27,371건 유지 (100.0%), 대조군 27,371건


M1: 장타 27,348/27,371건 유지 (99.9%), 대조군 27,348건


M2: 장타 27,156/27,371건 유지 (99.2%), 대조군 27,156건


## 6. M0 결과 확인

장타 쪽 값은 결정적이라 정확히 같아야 하고, 대조군은 무작위 추출이라 행 순서에 따라 조금 달라질 수 있습니다.
시즌 baseline 순수 감소 14.6%p [14.2, 15.1], 경기 내 baseline 15.6%p [15.1, 16.1], 재사용률 장타 45.5% vs 대조군 67.9%는
이 표본을 원본 데이터로 분석했을 때의 값이라, CSV만으로 다시 계산해도 그 범위 안에 들어오는지 확인합니다.
(점수차를 뺀 공식이라 혼합모델 오즈비는 비교 대상이 아닙니다.)

In [14]:
xbh0, placebo0 = pairs['M0']
assert len(xbh0) / n_eligible >= 0.999, 'M0에서 장타가 줄어듦: 대조군 후보가 부족한 칸이 있음'

season_m0 = pure_reduction(xbh0, placebo0, 'season_usage_rate')
game_m0 = pure_reduction(xbh0, placebo0, 'baseline_usage')
reuse_x = usable(xbh0, 'season_usage_rate')['reused_same_type'].mean()
reuse_p = usable(placebo0, 'season_usage_rate')['reused_same_type'].mean()
print(f'시즌 baseline 순수 감소 {season_m0["net"]:.4f} [{season_m0["ci_low"]:.4f}, {season_m0["ci_high"]:.4f}]  (기대 범위 [0.142, 0.151])')
print(f'경기 내 baseline 순수 감소 {game_m0["net"]:.4f} [{game_m0["ci_low"]:.4f}, {game_m0["ci_high"]:.4f}]  (기대 범위 [0.151, 0.161])')
print(f'재사용률 장타 {reuse_x:.3f} vs 대조군 {reuse_p:.3f}  (기대값 0.455 vs 0.679 안팎)')

assert 0.142 <= season_m0['net'] <= 0.151, season_m0
assert 0.151 <= game_m0['net'] <= 0.161, game_m0
assert abs(reuse_x - 0.455) < 0.002, reuse_x
assert abs(reuse_p - 0.679) < 0.010, reuse_p
print('M0 결과 확인 통과')

시즌 baseline 순수 감소 0.1439 [0.1394, 0.1484]  (기대 범위 [0.142, 0.151])
경기 내 baseline 순수 감소 0.1535 [0.1484, 0.1586]  (기대 범위 [0.151, 0.161])
재사용률 장타 0.455 vs 대조군 0.673  (기대값 0.455 vs 0.679 안팎)
M0 결과 확인 통과


## 7. 균형표 (표준화 평균차, SMD)

이벤트 시점 변수와 재대결 시점 변수(`*_next`)를 단계별로 비교합니다. 점수차는 데이터에 없어 점검하지 못합니다.
매칭에 쓴 변수(주자·아웃 등)는 M1·M2에서 0에 가까워지는 것이 정상이고, 매칭에 쓰지 않은 변수가 어떻게 움직이는지가 볼거리입니다.

In [15]:
BALANCE_COLUMNS = [
    'runners_n', 'risp', 'outs_when_up', 'inning', 'balls', 'strikes', 'platoon_match', 'runners_n_next', 'risp_next',
]
print('M0 평균 비교')
display(balance_table(*pairs['M0'], BALANCE_COLUMNS).round(3))

balance_df = pd.DataFrame({step: balance_table(*pairs[step], BALANCE_COLUMNS)['smd'] for step in STEPS}).round(3)
print('단계별 SMD')
display(balance_df)
print('|SMD| >= 0.1 인 항목:', {s: balance_df.index[balance_df[s].abs() >= 0.1].tolist() for s in STEPS})

M0 평균 비교


,xbh_mean,placebo_mean,smd
runners_n,0.486,0.401,0.124
risp,0.185,0.174,0.029
outs_when_up,0.877,0.905,-0.034
inning,2.390,2.448,-0.045
balls,1.116,1.086,0.029
strikes,1.007,1.103,-0.119
platoon_match,0.399,0.426,-0.056
runners_n_next,0.578,0.580,-0.003
risp_next,0.223,0.223,-0.000


단계별 SMD


,M0,M1,M2
runners_n,0.124,0.000,0.000
risp,0.029,0.000,0.000
outs_when_up,-0.034,0.000,0.000
inning,-0.045,-0.037,-0.031
balls,0.029,0.038,-0.037
strikes,-0.119,-0.104,-0.068
platoon_match,-0.056,-0.052,-0.041
runners_n_next,-0.003,-0.006,-0.010
risp_next,-0.000,-0.008,-0.009


|SMD| >= 0.1 인 항목: {'M0': ['runners_n', 'strikes'], 'M1': ['strikes'], 'M2': []}


## 8. 결과: 순수 감소와 혼합모델

* 혼합모델 공식은 `group * baseline_usage + balls + strikes + outs_when_up + stand + pitch_family + factor(season) + (0 + group | pitcher)`이며, CSV에 점수가 없어 `score_diff`는 뺐습니다. 이 공식을 모든 단계에 똑같이 적용하므로, 오즈비 자체보다 단계 사이의 변화를 비교하는 데 씁니다.
* C0는 M0 표본에 `runners_n`, `risp`를 공변량으로 더한 교차검증입니다(대조군은 그대로).

In [16]:
BASE_FORMULA = (
    'reused_same_type ~ group * baseline_usage + balls + strikes + outs_when_up '
    '+ stand + pitch_family + factor(season) + (0 + group | pitcher)'
)
C0_FORMULA = BASE_FORMULA.replace(' + (0 + group | pitcher)', ' + runners_n + risp + (0 + group | pitcher)')
REQUIRED = [
    'reused_same_type', 'baseline_usage', 'balls', 'strikes', 'outs_when_up', 'stand', 'pitch_family', 'season', 'pitcher',
]


def fit_mixed(xbh_: pd.DataFrame, placebo_: pd.DataFrame, formula: str, extra_required=()) -> dict:
    xf = xbh_[xbh_['has_next_ab'] & xbh_['baseline_usage'].notna()]
    pf = placebo_[placebo_['has_next_ab'] & placebo_['baseline_usage'].notna()]
    combined = build_combined_model_dataset(xf, pf, required_columns=REQUIRED + list(extra_required))
    fit_glmer(combined, formula)
    return {
        'n': len(combined), 'convergence': check_convergence(),
        'fe': extract_fixed_effects_table().set_index('term'),
    }


def key_terms(fit: dict) -> dict:
    fe = fit['fe']
    return {
        'group_or': fe.loc['group', 'odds_ratio'],
        'group_or_low': fe.loc['group', 'or_ci_low'],
        'group_or_high': fe.loc['group', 'or_ci_high'],
        'inter_or': fe.loc['group:baseline_usage', 'odds_ratio'],
        'inter_p': fe.loc['group:baseline_usage', 'p_value'],
    }

In [17]:
assert 'score_diff' not in BASE_FORMULA and 'score_diff' not in C0_FORMULA
assert C0_FORMULA.count('runners_n + risp') == 1 and 'runners_n' not in BASE_FORMULA
assert C0_FORMULA.endswith('(0 + group | pitcher)') and BASE_FORMULA.endswith('(0 + group | pitcher)')
print('공식 테스트 통과')

공식 테스트 통과


In [18]:
fits, rows = {}, []
for step, (xbh_m, placebo_m) in pairs.items():
    fits[step] = fit_mixed(xbh_m, placebo_m, BASE_FORMULA)
    season, game = pure_reduction(xbh_m, placebo_m, 'season_usage_rate'), pure_reduction(xbh_m, placebo_m, 'baseline_usage')
    rows.append({
        'step': step, 'n_xbh': len(xbh_m), 'retention': len(xbh_m) / n_eligible, 'n_model': fits[step]['n'],
        'net_season': season['net'], 'season_low': season['ci_low'], 'season_high': season['ci_high'],
        'net_game': game['net'], 'game_low': game['ci_low'], 'game_high': game['ci_high'],
        **key_terms(fits[step]),
    })

# C0: M0 표본 + 주자 공변량 (순수 감소는 M0와 같은 표본이므로 M0 값을 그대로 쓴다)
fits['C0'] = fit_mixed(*pairs['M0'], C0_FORMULA, extra_required=('runners_n', 'risp'))
rows.append({**rows[0], 'step': 'C0', 'n_model': fits['C0']['n'], **key_terms(fits['C0'])})

summary = pd.DataFrame(rows).set_index('step')
display(summary.round(4))
print({step: fit['convergence'] for step, fit in fits.items()})
summary.to_csv('../data/processed/situation_matched_ladder_summary.csv')

2026-09-27 17:27:23,514 [WARNING] R callback write-console: Loading required package: Matrix
  


,n_xbh,retention,n_model,net_season,season_low,season_high,net_game,game_low,game_high,group_or,group_or_low,group_or_high,inter_or,inter_p
step,,,,,,,,,,,,,,
M0,27371,1.0000,53800,0.1439,0.1394,0.1484,0.1535,0.1484,0.1586,0.3042,0.2825,0.3276,1.7568,0.0
M1,27348,0.9992,53788,0.1450,0.1405,0.1494,0.1534,0.1483,0.1585,0.2818,0.2616,0.3035,1.9986,0.0
M2,27156,0.9921,53393,0.1441,0.1396,0.1486,0.1514,0.1463,0.1566,0.2929,0.2719,0.3156,1.8960,0.0
C0,27371,1.0000,53800,0.1439,0.1394,0.1484,0.1535,0.1484,0.1586,0.3027,0.2811,0.3260,1.7630,0.0


{'M0': 'OK: no convergence warnings; isSingular=False', 'M1': 'OK: no convergence warnings; isSingular=False', 'M2': 'OK: no convergence warnings; isSingular=False', 'C0': 'OK: no convergence warnings; isSingular=False'}


## 9. 주자 상황별 층화 순수 감소

같은 주자 상황 안에서도 장타와 대조군의 차이가 남는지 봅니다. 3단계는 서로 겹치지 않습니다:
주자 없음 / 1루에만 주자 / 득점권(2·3루 주자, 1루 동시 포함). 각 칸에 양쪽 30건 미만이면 건너뜁니다.

In [19]:
def stratified_reduction(xbh_: pd.DataFrame, placebo_: pd.DataFrame, baseline_col: str, min_n: int = 30) -> pd.DataFrame:
    x, p = usable(xbh_, baseline_col), usable(placebo_, baseline_col)
    x['runner_group'], p['runner_group'] = runner_group(x), runner_group(p)
    rows = []
    for level in ['주자 없음', '1루만', '득점권']:
        gx, gp = x[x['runner_group'] == level], p[p['runner_group'] == level]
        if len(gx) < min_n or len(gp) < min_n:
            continue
        r = summarize_diff_in_diff(gx['diff'], gp['diff'])
        rows.append({
            'runner_group': level, 'n_xbh': len(gx), 'n_placebo': len(gp),
            'net': r['net_effect'], 'ci_low': r['ci_low'], 'ci_high': r['ci_high'],
        })
    return pd.DataFrame(rows)


strat = pd.concat(
    [
        stratified_reduction(*pairs[step], baseline).assign(step=step, baseline=baseline)
        for step in STEPS
        for baseline in ('season_usage_rate', 'baseline_usage')
    ],
    ignore_index=True,
)
display(strat.pivot_table(index=['baseline', 'runner_group'], columns='step', values='net').round(4))
display(strat.round(4))
strat.to_csv('../data/processed/situation_matched_stratified_by_runners.csv', index=False)

step                                M0      M1      M2
baseline          runner_group                        
baseline_usage    1루만           0.1535  0.1489  0.1406
                  득점권           0.1553  0.1541  0.1526
                  주자 없음         0.1545  0.1546  0.1544
season_usage_rate 1루만           0.1413  0.1395  0.1335
                  득점권           0.1536  0.1487  0.1485
                  주자 없음         0.1423  0.1455  0.1460

,runner_group,n_xbh,n_placebo,net,ci_low,ci_high,step,baseline
0,주자 없음,17229,18837,0.1423,0.1368,0.1478,M0,season_usage_rate
1,1루만,5067,3765,0.1413,0.1299,0.1526,M0,season_usage_rate
2,득점권,5075,4769,0.1536,0.1431,0.1642,M0,season_usage_rate
3,주자 없음,16718,18430,0.1545,0.1481,0.1609,M0,baseline_usage
4,1루만,5063,3761,0.1535,0.1411,0.1658,M0,baseline_usage
5,득점권,5065,4763,0.1553,0.1440,0.1665,M0,baseline_usage
6,주자 없음,17229,17229,0.1455,0.1399,0.1511,M1,season_usage_rate
7,1루만,5067,5067,0.1395,0.1291,0.1499,M1,season_usage_rate
8,득점권,5052,5052,0.1487,0.1383,0.1592,M1,season_usage_rate
9,주자 없음,16718,16860,0.1546,0.1480,0.1612,M1,baseline_usage


## 10. 해석

* **M0가 기대 범위 안에 있는지.** 시즌 baseline 순수 감소 14.39%p [13.94, 14.84] (기대 범위 [14.2, 15.1]), 경기 내 baseline 15.35%p [14.84, 15.86] (기대 범위 [15.1, 16.1])로 범위 안에는 들었지만 점 추정치가 정확히 같지는 않습니다.
  대조군이 무작위 추출이라 CSV의 행 순서가 원본 데이터와 달라 생긴 차이로 보입니다. 혼합모델 `group` 오즈비는 0.304로, 점수차를 포함한 모델의 오즈비와는 다르며, 점수차를 뺀 공식과 다른 대조군 표본 중 무엇 때문인지는 분리하지 못했습니다. 그래서 이후 비교는 이 노트북 안(M0~C0 단계 사이)에서만 합니다.
* **지적사항은 실제로 있었습니다.** M0에서 이벤트 시점 주자 수의 SMD가 0.124(장타 0.486명 vs 대조군 0.401명)로 장타가 주자 있는 상황에서 더 자주 나왔고, 스트라이크 수도 −0.119였습니다.
  M1에서 주자·아웃 SMD가 0.000이 되었고, M2에서는 균형표의 모든 항목이 |SMD| < 0.1(최대 0.068, 스트라이크)입니다. 재대결 시점의 주자 변수는 M0부터 균형이었습니다(|SMD| ≤ 0.010).
  유지율은 M1 99.9%, M2 99.2%로 장타 표본이 거의 줄지 않았습니다.
* **그런데 결과는 거의 변하지 않았습니다.** 시즌 baseline 순수 감소는 M0 14.39%p → M1 14.50%p → M2 14.41%p, 경기 내 baseline은 15.35 → 15.34 → 15.14%p로, 변화가 M0 신뢰구간 폭(약 ±0.5%p) 안입니다.
  `group` 오즈비도 0.304 [0.283, 0.328] → 0.282 [0.262, 0.304] → 0.293 [0.272, 0.316]으로 신뢰구간이 서로 겹칩니다. 교차검증 C0(대조군은 M0 그대로, 주자 공변량 추가)도 0.303 [0.281, 0.326]으로 같습니다.
  `group × baseline_usage` 오즈비도 1.76 → 2.00 → 1.90(C0 1.76)으로 모두 1보다 큽니다. → **관측한 주자·아웃·이닝·카운트를 맞춰도 장타 후 구종 회피는 남습니다.**
* **주자 상황별로도 남습니다.** 같은 주자 상황 안에서도 순수 감소가 모두 양수이고 신뢰구간이 0을 벗어납니다. M2 시즌 baseline 기준으로 주자 없음 14.60%p, 1루에만 13.35%p, 득점권 14.85%p입니다.
  세 층의 신뢰구간이 서로 겹쳐서 위기 상황일수록 효과가 달라진다는 뚜렷한 증거는 없습니다.
* **한계.**
  * 점수차는 CSV에 없어 통제하지 못했습니다. 점수차는 위기 판단의 일부입니다.
  * 레버리지, 투수 피로·압박 같은 미관측 요인은 배제하지 못했습니다. 이 결과는 "관측 가능한 위기 지표를 맞춰도 남는다"이지 "위기의 영향이 없다"는 증명이 아닙니다.
  * 주자 값은 투구 직전 상태이며 타석 도중 도루 등으로 바뀐 상태는 반영하지 않습니다.
  * M0의 점 추정치가 기대 범위와 완전히 일치하지 않는 점(위 첫 항목)도 감안해서 읽어야 합니다.